# 10 — Single-column figure layout

This notebook changes figure layout only. It does not alter analytical data. `demo` reads notebook 07 synthetic aggregates; `public-results` reads only `data/public/figure_data`. No paper/review microdata, models, or statistical definitions are touched. PNG and PDF are written to mode-separated `figures/single_column` directories.

SOURCE_LINEAGE_LITERAL: `redraw_fig4_fig5_single_column_v2.ipynb` cells 0–5. The notebook magic and interactive display calls were removed; colors, aspect/country order and sizes follow the source.


In [ ]:
from pathlib import Path
import sys
import os
import yaml

ROOT = Path.cwd().resolve()
if not (ROOT / "src" / "project_paths.py").exists():
    ROOT = ROOT.parent
if not (ROOT / "src" / "project_paths.py").exists():
    raise FileNotFoundError("Run from the release root or its notebooks directory.")
sys.path.insert(0, str(ROOT))
from src.project_paths import project_path

with (ROOT / "config" / "config.example.yaml").open(encoding="utf-8") as fh:
    CONFIG = yaml.safe_load(fh)
MODE = os.environ.get("NC_RUNTIME_MODE", CONFIG.get("runtime", {}).get("mode", "demo"))
if MODE not in ('demo', 'public-results'):
    raise ValueError("This notebook requires runtime.mode in: demo, public-results")
ANALYSIS = CONFIG["analysis"]
if int(ANALYSIS["formal_country_min_papers"]) != 500:
    raise ValueError("The formal country threshold must remain 500.")
if int(ANALYSIS["large_team_cutoff"]) != 50:
    raise ValueError("The formal large-team cutoff must remain 50.")
if ANALYSIS["country_reference"] != "US" or ANALYSIS["gender_reference"] != "male":
    raise ValueError("Formal model references must remain US and male.")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm
if MODE == "demo":
    IN_DIR = project_path("results/demo/strict/tables")
    OUT_DIR = project_path("results/demo/figures/single_column")
else:
    IN_DIR = project_path(CONFIG["paths"]["public_results_data_dir"]) / "figure_data"
    OUT_DIR = project_path("results/regenerated/figures/single_column")
required = ["descriptive_per_aspect_disagreement_main_strict.csv", "figure_data_country_aspect_deviation_pp.csv", "figure_data_gender_aspect_deviation_pp.csv"]
missing = [name for name in required if not (IN_DIR / name).is_file()]
if missing:
    raise FileNotFoundError(f"Required aggregate inputs are absent in {IN_DIR}: {missing}")
for name in required:
    cols = set(pd.read_csv(IN_DIR / name, nrows=1).columns)
    if cols.intersection({"paper_code", "review_id", "review_text", "author_name", "paper_title"}):
        raise ValueError(f"Non-aggregate input rejected: {name}")
    expected = {"aspect_label", "mean_mapd"} if name.startswith("descriptive_") else {"group", "Methodology", "Results", "Experiments and Evaluation", "Data and Datasets", "Figures and Tables", "Writing, Language and Terminology", "Theory and Models", "Comparison and Context", "Paper Assessment", "Novelty and Impact", "Validity, Rigor and Quality", "Limitations"}
    if not expected.issubset(cols):
        raise ValueError(f"Aggregate input schema incomplete: {name}")
    allowed = {"aspect_class", "aspect_label", "n_papers", "mean_mapd", "median_mapd", "sd_mapd", "q25_mapd", "q75_mapd"} if name.startswith("descriptive_") else expected
    if cols.difference(allowed):
        raise ValueError(f"Unexpected columns in aggregate input: {name}: {sorted(cols.difference(allowed))}")
OUT_DIR.mkdir(parents=True, exist_ok=True)
plt.rcParams["font.family"] = "DejaVu Sans"
plt.rcParams["axes.unicode_minus"] = False
plt.rcParams["pdf.fonttype"] = 42
plt.rcParams["ps.fonttype"] = 42

# The 12 aspects and the labels used by Figure 4 (main analysis notebook).
ASPECT_ORDER = [
    "Methodology",
    "Results",
    "Experiments and Evaluation",
    "Data and Datasets",
    "Figures and Tables",
    "Writing, Language and Terminology",
    "Theory and Models",
    "Comparison and Context",
    "Paper Assessment",
    "Novelty and Impact",
    "Validity, Rigor and Quality",
    "Limitations",
]

ASPECT_LABEL = {
    "Methodology": "Methodology",
    "Results": "Results",
    "Experiments and Evaluation": "Experiments & evaluation",
    "Data and Datasets": "Data & datasets",
    "Figures and Tables": "Figures & tables",
    "Writing, Language and Terminology": "Writing & language",
    "Theory and Models": "Theory & models",
    "Comparison and Context": "Comparison & context",
    "Paper Assessment": "Paper assessment",
    "Novelty and Impact": "Novelty & impact",
    "Validity, Rigor and Quality": "Validity & rigor",
    "Limitations": "Limitations",
}

COUNTRY_ORDER = ["US", "CN", "DE", "GB", "JP", "FR", "KR", "CA", "CH", "AU", "NL", "ES", "SE"]
GENDER_ORDER = ["male", "female"]


def save_figure(fig, name: str) -> None:
    fig.savefig(OUT_DIR / f"{name}.png", dpi=300, bbox_inches="tight")
    fig.savefig(OUT_DIR / f"{name}.pdf", bbox_inches="tight")
    print(f"Saved: {OUT_DIR / (name + '.png')}")
    print(f"Saved: {OUT_DIR / (name + '.pdf')}")


# =========================================================


In [ ]:
# 1. Figure 4: per-aspect disagreement (landscape ~16:9)
# =========================================================
mapd = pd.read_csv(IN_DIR / "descriptive_per_aspect_disagreement_main_strict.csv")
plot_mapd = mapd.sort_values("mean_mapd", ascending=True).copy()

fig, ax = plt.subplots(figsize=(7.2, 4.05))  # 16:9
ax.barh(plot_mapd["aspect_label"], plot_mapd["mean_mapd"] * 100,
        color="#4C78A8", edgecolor="white", linewidth=0.4)
ax.set_xlabel("Mean pairwise absolute difference (pp)", fontsize=9)
ax.set_ylabel("")
ax.tick_params(axis="y", labelsize=8)
ax.tick_params(axis="x", labelsize=8)
for i, v in enumerate(plot_mapd["mean_mapd"] * 100):
    ax.text(v, i, f" {v:.2f}", va="center", fontsize=7.5)
ax.margins(x=0.02)
for s in ["top", "right"]:
    ax.spines[s].set_visible(False)

save_figure(fig, "fig_per_aspect_disagreement_overall")
plt.close(fig)


# =========================================================


In [ ]:
# 2. Figure 5: aspect deviation heatmaps, rotated 90 degrees
#    12 aspects = shared y-axis; countries / genders = x-axis
# =========================================================
country_df = pd.read_csv(IN_DIR / "figure_data_country_aspect_deviation_pp.csv")
gender_df = pd.read_csv(IN_DIR / "figure_data_gender_aspect_deviation_pp.csv")

country_mat = (
    country_df.set_index("group")
    .reindex(index=COUNTRY_ORDER)
    .reindex(columns=ASPECT_ORDER)
)
gender_mat = (
    gender_df.set_index("group")
    .reindex(index=GENDER_ORDER)
    .reindex(columns=ASPECT_ORDER)
)
country_mat = country_mat.apply(pd.to_numeric, errors="coerce").fillna(0)
gender_mat = gender_mat.apply(pd.to_numeric, errors="coerce").fillna(0)

# Rotate: aspects become the rows (shared y-axis).
country_rot = country_mat.T  # 12 aspects x 13 countries
gender_rot = gender_mat.T    # 12 aspects x 2 genders
ylabels = [ASPECT_LABEL[a] for a in ASPECT_ORDER]

all_values = np.concatenate([
    country_rot.to_numpy().ravel(),
    gender_rot.to_numpy().ravel(),
])
max_abs = np.nanmax(np.abs(all_values))
max_abs = np.ceil(max_abs * 10) / 10
if max_abs == 0:
    max_abs = 1.0

norm = TwoSlopeNorm(vmin=-max_abs, vcenter=0, vmax=max_abs)
cmap = "RdBu_r"


def draw_heatmap_rotated(ax, mat, panel_label, show_ylabels=True,
                         annot_fmt="{:.1f}", annot_fontsize=6.0,
                         xlabel_fontsize=7.0):
    im = ax.imshow(mat.to_numpy(), cmap=cmap, norm=norm,
                   aspect="auto", interpolation="nearest")
    n_rows, n_cols = mat.shape

    ax.set_xticks(np.arange(n_cols))
    ax.set_xticklabels(mat.columns.tolist(), fontsize=xlabel_fontsize)
    ax.set_yticks(np.arange(n_rows))
    if show_ylabels:
        ax.set_yticklabels(ylabels, fontsize=6.8)
    else:
        ax.set_yticklabels([])

    ax.set_xticks(np.arange(-0.5, n_cols, 1), minor=True)
    ax.set_yticks(np.arange(-0.5, n_rows, 1), minor=True)
    ax.grid(which="minor", color="white", linestyle="-", linewidth=0.75)
    ax.tick_params(which="minor", bottom=False, left=False)
    ax.tick_params(axis="y", length=0)
    ax.set_ylabel("")

    threshold = max_abs * 0.55
    for i in range(n_rows):
        for j in range(n_cols):
            value = mat.iat[i, j]
            if np.isfinite(value):
                text_color = "white" if abs(value) >= threshold else "black"
                ax.text(j, i, annot_fmt.format(value),
                        ha="center", va="center",
                        fontsize=annot_fontsize, color=text_color)

    for spine in ax.spines.values():
        spine.set_linewidth(0.7)

    ax.text(-0.02, 1.02, panel_label, transform=ax.transAxes,
            ha="left", va="bottom", fontsize=8.2, fontweight="bold")
    return im


fig = plt.figure(figsize=(7.6, 4.1))  # landscape, ~16:9
gs = fig.add_gridspec(
    1, 2,
    width_ratios=[country_rot.shape[1], gender_rot.shape[1]],
    wspace=0.05,
    left=0.20, right=0.84, top=0.90, bottom=0.08,
)
ax_country = fig.add_subplot(gs[0, 0])
ax_gender = fig.add_subplot(gs[0, 1])
# Note: do NOT use sharey here - with a shared y-axis the second panel's
# set_yticklabels([]) would wipe the aspect labels of the first panel too.

im = draw_heatmap_rotated(
    ax_country, country_rot, "(a) First-author country/region",
    show_ylabels=True,
)
draw_heatmap_rotated(
    ax_gender, gender_rot, "(b) Inferred first-author gender",
    show_ylabels=False,
)

cax = fig.add_axes([0.865, 0.14, 0.020, 0.70])
cb = fig.colorbar(im, cax=cax)
cb.set_label("Difference from overall share (pp)", fontsize=7.5)
cb.ax.tick_params(labelsize=6.5)

save_figure(fig, "fig_aspect_deviation_combined_rotated_v2")
plt.close(fig)

print("\nDone. Outputs written to:", OUT_DIR)
